# Verify frozen source, slot and review coverage and hashes


Verify actual recorded artifacts and all used response checksums. These checks establish reconciliation and traceability, not biological admission or literature completeness.

In [1]:
from pathlib import Path
import json,hashlib,pandas as pd
p = Path('results/dataset_selection_2026_10_07')
manifest = json.loads((p/'manifest.json').read_text())
review = json.loads((p/'review_manifest.json').read_text())
for m in (manifest, review):
    for file, digest in m['input_sha256'].items():
        assert hashlib.sha256(Path(file).read_bytes()).hexdigest() == digest, file
    for file, digest in m['outputs'].items():
        assert hashlib.sha256((p/file).read_bytes()).hexdigest() == digest, file
for file, digest in manifest['used_request_sha256'].items():
    assert hashlib.sha256((p/'requests'/file).read_bytes()).hexdigest() == digest, file
len(manifest["used_request_sha256"])

459

In [2]:
sources = pd.read_csv(p/'source_publications.csv')
slots = pd.read_csv(p/'slot_comparisons.csv')
queue = pd.read_csv(p/'source_review_queue.csv')
searches = pd.read_csv(p/'slot_searches.csv')
literature = json.loads((p/'literature_candidates.json').read_text())
assert len(sources) == 162 == sources.dataset_id.nunique() == len(queue)
assert len(slots) == 297 == slots.slot.nunique()
assert len(searches) == 594 and searches.status.eq("ok").all()
assert not slots.change_proposed.any() and not slots.rankable_groups.any()
assert all(r["admission"] == "pending_sample_and_deposit_review" and r["selection_score"] is None and r["comprehensiveness"] is None for r in literature)
assert all(c['score'] is None and c['admission'] == 'pending' for row in slots.candidate_factors for c in json.loads(row))
assert sources.citations.notna().sum() == 95 and sources.citations.isna().sum() == 67
assert not queue.literature_best_verified.any()
manifest["summary"]

{'diagnostic_flags': 6, 'literature_rows': 5058, 'promoted_sources': 0, 'provisional_changes': 0, 'publication_statuses': {'accession_publication_not_found': 4, 'accession_publications_need_review': 6, 'derived_or_local_no_own_publication': 13, 'reference_resource_publication_unresolved': 19, 'resolved_recorded_doi': 12, 'resolved_registry_pmid': 83, 'unresolved_publication': 25}, 'search_failures': 0, 'search_jobs': 432, 'slots': 297, 'source_count': 162, 'unique_search_queries': 216}

In [3]:
pd.read_csv(p/'host_progress.csv')

,parasite,filled_slot_views,total_slot_views,host_gene_space_status,inference_pack_status
0,Pf,4,24,pending,pending
1,Tg,8,26,pending,pending
